# 02 · Planner Selection

> Part of the **ICAPS 2026 Planning Ontology Tutorial**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ai4society/ICAPS26-planning-ontology-tutorial/blob/main/notebooks/02_planner_selection.ipynb)

Many planners can solve the same domain, and their performance varies from one
domain to the next. Once you record per-domain planner relevance in the knowledge
graph, a single SPARQL query answers a practical question: **which planner should
I run on this domain?**

In this notebook you:

- **Rank** the planners for `blocksworld` by recorded relevance
- **Pick** the top recommendation with one query
- **Repeat** the same query against the real IPC-2018 graph

> **Tip.** The **Core** path needs only `rdflib` and `pandas`. **Go deeper** writes
> custom SPARQL and runs a live planner through the optional Unified Planning
> library.

---

## Setup

Run this first. In **Colab** it installs the dependencies and fetches the tutorial
data. **Locally** it uses your `requirements.txt` environment.

In [1]:
# In Colab this installs dependencies and fetches the tutorial data.
# Locally it assumes you installed requirements.txt.
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import subprocess
    subprocess.run(["pip", "install", "-q", "rdflib", "pandas"], check=True)
    subprocess.run(["git", "clone", "-q",
                    "https://github.com/ai4society/ICAPS26-planning-ontology-tutorial.git"],
                   check=False)
    DATA = Path("ICAPS26-planning-ontology-tutorial/data")
else:
    DATA = Path("..") / "data"

print("data directory:", DATA.resolve())

data directory: /Users/nitingupta/usc/ai4s/conferences/icaps26/tutorial/planning-ontology-tutorial/data


---
## 1. Load the prebuilt knowledge graph

The tutorial ships a small blocksworld KG with planner relevance recorded. You load
it once and reuse it for every query below. The relevance values are curated for the
tutorial; section 4 runs the same query on the real IPC-2018 results.

In [2]:
from rdflib import Graph, Namespace, Literal, RDF, RDFS, XSD

PO = Namespace("https://purl.org/ai4s/ontology/planning#")
PREFIX = (
    "PREFIX po: <https://purl.org/ai4s/ontology/planning#> "
    "PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#> "
)

kg = Graph()
kg.parse(str(DATA / "kgs" / "blocksworld_tutorial.ttl"), format="turtle")
print(f"loaded blocksworld KG: {len(kg)} triples")

loaded blocksworld KG: 106 triples


---
## 2. Rank the planners for a domain (Core)

The ontology records planner relevance in three buckets:
`hasHighRelevancePlanner`, `hasMediumRelevancePlanner`, and
`hasLowRelevancePlanner`. One query reads all three and sorts them high to low, so
you see the full ranking.

> **Note.** The paper's prose abbreviates these properties to `hasHighRelevance`.
> The published ontology uses the longer `...Planner` names, so the queries here use
> those.

In [3]:
import pandas as pd

rank_query = PREFIX + """
SELECT ?relevance ?planner WHERE {
  VALUES (?prop ?relevance ?order) {
    (po:hasHighRelevancePlanner   "high"   1)
    (po:hasMediumRelevancePlanner "medium" 2)
    (po:hasLowRelevancePlanner    "low"    3)
  }
  po:blocksworld ?prop ?p .
  ?p rdfs:label ?planner .
} ORDER BY ?order ?planner
"""

rows = [(str(r.relevance), str(r.planner)) for r in kg.query(rank_query)]
pd.DataFrame(rows, columns=["relevance", "planner"])

,relevance,planner
0,high,Fast Downward
1,high,LAMA-2011
2,medium,BFWS
3,low,Probe


---
## 3. Pick the top recommendation (Core)

To return a single answer, you fall back through the buckets with `COALESCE`: take
a high-relevance planner if one exists, otherwise a medium one, otherwise a low one.
`ORDER BY ... LIMIT 1` makes the choice deterministic.

In [4]:
best_query = PREFIX + """
SELECT ?planner WHERE {
  OPTIONAL { po:blocksworld po:hasHighRelevancePlanner   ?h . ?h rdfs:label ?hl }
  OPTIONAL { po:blocksworld po:hasMediumRelevancePlanner ?m . ?m rdfs:label ?ml }
  OPTIONAL { po:blocksworld po:hasLowRelevancePlanner    ?l . ?l rdfs:label ?ll }
  BIND(COALESCE(?hl, ?ml, ?ll) AS ?planner)
} ORDER BY ?planner LIMIT 1
"""

best = [str(r.planner) for r in kg.query(best_query)]
print("recommended planner for blocksworld:", best[0] if best else "none found")

recommended planner for blocksworld: Fast Downward


---
## 4. The same query on real IPC-2018 data

The recommendation logic does not depend on the tutorial data. Here you point it at
a slice of the **real IPC-2018 knowledge graph**, which covers the `agricola` and
`caldera` domains.

> **Working across ontology versions.** The IPC-2018 instance data uses an earlier
> namespace, so you swap the `po:` prefix for that version's IRI. The property names
> are identical, and RDFLib loads either graph the same way.

In [5]:
OLD = "http://www.semanticweb.org/muppa/ontologies/2022/4/plan-ontology#"
PREFIX_OLD = (
    f"PREFIX po: <{OLD}> "
    "PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#> "
)

real = Graph()
real.parse(str(DATA / "kgs" / "ipc2018_planner_info_sample.ttl"), format="turtle")
print(f"real IPC-2018 subset: {len(real)} triples")

counts_query = PREFIX_OLD + """
SELECT ?domain ?relevance (COUNT(?p) AS ?n) WHERE {
  VALUES (?prop ?relevance) {
    (po:hasHighRelevancePlanner   "high")
    (po:hasMediumRelevancePlanner "medium")
    (po:hasLowRelevancePlanner    "low")
  }
  ?d ?prop ?p .
  ?d rdfs:label ?domain .
} GROUP BY ?domain ?relevance
"""

rows = [(str(r.domain), str(r.relevance), int(r.n)) for r in real.query(counts_query)]
counts = (pd.DataFrame(rows, columns=["domain", "relevance", "planners"])
          .pivot(index="domain", columns="relevance", values="planners")
          .fillna(0).astype(int))[["high", "medium", "low"]]
counts

real IPC-2018 subset: 56 triples


relevance,high,medium,low
domain,,,
agricola,5,4,9
caldera,8,8,2


The same `COALESCE` recommendation works per domain. Here are the
high-relevance planners the query would draw from for `caldera`.

In [6]:
high_caldera = PREFIX_OLD + """
SELECT ?planner WHERE {
  ?d rdfs:label "caldera" .
  ?d po:hasHighRelevancePlanner ?p .
  ?p rdfs:label ?planner .
} ORDER BY ?planner
"""
print("high-relevance planners for caldera:")
for r in real.query(high_caldera):
    print(" ", r.planner)

high-relevance planners for caldera:
  Complementary2
  Delfi1
  FDMS1
  FDMS2
  Metis1
  Metis2
  Planning_PDBs
  Scorpion


---
## Go deeper: parameterize the recommendation

Wrapping the ranking query in a function lets you ask about any domain by label.
Notebook 00 hides this logic behind a one-line helper.

In [7]:
def rank_planners(graph, domain_label, prefix):
    q = prefix + """
    SELECT ?relevance ?planner WHERE {
      VALUES (?prop ?relevance ?order) {
        (po:hasHighRelevancePlanner   "high"   1)
        (po:hasMediumRelevancePlanner "medium" 2)
        (po:hasLowRelevancePlanner    "low"    3)
      }
      ?d rdfs:label "%s" .
      ?d ?prop ?p .
      ?p rdfs:label ?planner .
    } ORDER BY ?order ?planner
    """ % domain_label
    return [(str(r.relevance), str(r.planner)) for r in graph.query(q)]

for relevance, planner in rank_planners(real, "agricola", PREFIX_OLD):
    print(f"  {relevance:<6} {planner}")

  high   Complementary1
  high   Delfi1
  high   Delfi2
  high   FDMS2
  high   symb_Bi_dir
  medium Complementary2
  medium FDMS1
  medium MSP
  medium Planning_PDBs
  low    Blind
  low    DecStar
  low    Metis1
  low    Metis2
  low    Scorpion
  low    Symple_1
  low    Symple_2
  low    maplan_1
  low    maplan_2


---
## Go deeper: run a planner and write results back (optional, third-party)

This step runs an actual PDDL planner on the blocksworld problem and adds the fresh
result to the KG. It uses **Unified Planning** (aiplan4eu) with the pure-Python
**pyperplan** engine. Unified Planning is a third-party library, not part of the
ontology, so the cell installs it on demand in Colab and skips itself when the
library is absent.

In [8]:
# Optional. Skips cleanly if Unified Planning is not installed.
try:
    if IN_COLAB:
        import subprocess
        subprocess.run(["pip", "install", "-q", "unified-planning", "up-pyperplan"],
                       check=True)
    from unified_planning.io import PDDLReader
    from unified_planning.shortcuts import OneshotPlanner
    from unified_planning.shortcuts import get_environment
    get_environment().credits_stream = None
    HAVE_UP = True
except Exception as exc:
    HAVE_UP = False
    print("Unified Planning not available, skipping the live run.")
    print("To enable it: pip install unified-planning up-pyperplan")

if HAVE_UP:
    reader = PDDLReader()
    problem = reader.parse_problem(
        str(DATA / "domains" / "blocksworld" / "domain.pddl"),
        str(DATA / "domains" / "blocksworld" / "problem.pddl"),
    )
    with OneshotPlanner(name="pyperplan") as planner:
        result = planner.solve(problem)

    plan = result.plan
    actions = list(plan.actions) if plan else []
    print(f"engine: {result.engine_name}")
    print(f"plan length: {len(actions)} actions")
    for step in actions:
        print("  ", step)

    # Write the fresh plan back into the KG under the po: vocabulary.
    fresh = PO["plan_pyperplan"]
    kg.add((fresh, RDF.type, PO.Plan))
    kg.add((PO.problem_3_1, PO.hasPlan, fresh))
    kg.add((fresh, PO.isGeneratedBy, PO.pyperplan))
    kg.add((fresh, PO.hasPlanCost,
            Literal(len(actions), datatype=XSD.nonNegativeInteger)))
    print(f"\nKG now holds {len(kg)} triples (added the pyperplan plan).")

engine: Pyperplan
plan length: 6 actions
   unstack(b3, b1)
   put-down(b3)
   pick-up(b2)
   stack(b2, b1)
   pick-up(b3)
   stack(b3, b2)

KG now holds 110 triples (added the pyperplan plan).


---
## Recap and next

You ranked planners for a domain, picked a single recommendation with `COALESCE`,
and ran the same query against the real IPC-2018 graph. The optional step solved the
problem with a live planner and wrote the result back into the KG.

| Next notebook | Focus |
| --- | --- |
| **03 · Plan explanation** | Turn a plan into a readable narrative |
| **00 · Quickstart** | Ask the same questions through one-line helpers |